# News Search Engine — Notebook kiểm thử chi tiết

Kiểm tra **từng chức năng** (F-02 → F-15), tầng **API**, và **benchmark** lexical/dense/hybrid,
in ra chi tiết để quan sát hành vi.

> Chạy tuần tự từ trên xuống. Kernel: Python của `.venv` (đã cài fastapi, numpy, pytest...).
> Thư mục làm việc = gốc dự án `news-search-engine` (để `import news_search` và đọc `data/`).

## 0. Thiết lập — nạp dữ liệu mẫu, dựng IndexManager + SearchPipeline

In [1]:
import os, sys, json
# Tìm gốc dự án (thư mục chứa news_search/) rồi chdir + thêm vào sys.path,
# để notebook chạy được dù đặt trong notebooks/.
_d = os.getcwd()
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "news_search")):
    _d = os.path.dirname(_d)
os.chdir(_d); sys.path.insert(0, _d)
from datetime import datetime, timedelta, timezone
from news_search.config import Settings
from news_search.index.manager import IndexManager
from news_search.search.pipeline import SearchPipeline
from news_search.models import SearchQuery, SearchFilters, Article, Entity

def h(title):
    print("=" * 74); print(title); print("=" * 74)

NOW = datetime(2026, 7, 7, 12, 0, 0, tzinfo=timezone(timedelta(hours=7)))
# Demo deterministic & offline: ép hash/local (production dùng .env: bge/milvus)
settings = Settings(embedder="hash", vector_backend="local")
manager = IndexManager(settings)
raws = json.load(open("data/sample_articles.json", encoding="utf-8"))
n = manager.bulk_index(raws)
pipeline = SearchPipeline(manager, settings)
print(f"Đã index {n} bài viết.")
print(f"Embedder = {settings.embedder} (dim={manager.embedder.dim}) | "
      f"vector_backend = {settings.vector_backend} | reranker = {settings.reranker}")
print(f"half_life(thường)={settings.half_life_days}d, fresh={settings.fresh_half_life_days}d, "
      f"floor={settings.decay_floor}")

Đã index 29 bài viết.
Embedder = hash (dim=256) | vector_backend = local | reranker = none
half_life(thường)=30.0d, fresh=2.0d, floor=0.5


## 1. F-02 / F-03 — Ingest: làm sạch HTML & token hóa tiếng Việt

In [2]:
from news_search.ingest.cleaner import clean_html, normalize_article
from news_search.ingest.tokenizer import normalize_text, fold_diacritics, tokenize, fold_tokens

h("clean_html — bỏ script/style, giữ ngắt đoạn, unescape entity")
raw_html = ('<div><script>alert(1)</script><style>.x{}</style>'
            '<h1>Giá vàng</h1><p>Đoạn 1 &amp; còn nữa.</p><p>Đoạn 2.</p></div>')
print("HTML gốc :", raw_html)
print("Sau sạch :", repr(clean_html(raw_html)))

h("normalize_article — chuẩn hóa dict thô -> Article (naive datetime -> +07)")
art = normalize_article({
    "article_id": " demo-1 ", "title": "  Giá vàng tăng  ",
    "body": "<p>Nội dung <b>quan trọng</b> &amp; chi tiết.</p>",
    "url": "/kt/demo-1", "published_at": "2026-07-07T09:00:00",
    "author": " An ", "category": "Kinh tế",
})
print("article_id  :", repr(art.article_id))
print("title       :", repr(art.title))
print("body(sạch)  :", repr(art.body))
print("published_at:", art.published_at.isoformat(), "(naive -> gán UTC+7)")
print("author/cat  :", art.author, "/", art.category)

h("tokenizer — tiếng Việt: normalize / fold dấu / tokenize")
s = "Lạm phát ở Đồng Tháp TĂNG mạnh!"
print("gốc          :", s)
print("normalize    :", normalize_text(s))
print("fold_diacrit :", fold_diacritics(s), "  (Đồng -> Dong)")
print("tokenize     :", tokenize(s))
print("fold_tokens  :", fold_tokens(tokenize(s)), "  (dùng cho match không dấu)")

clean_html — bỏ script/style, giữ ngắt đoạn, unescape entity
HTML gốc : <div><script>alert(1)</script><style>.x{}</style><h1>Giá vàng</h1><p>Đoạn 1 &amp; còn nữa.</p><p>Đoạn 2.</p></div>
Sau sạch : 'Giá vàng\nĐoạn 1 & còn nữa.\nĐoạn 2.'
normalize_article — chuẩn hóa dict thô -> Article (naive datetime -> +07)
article_id  : 'demo-1'
title       : 'Giá vàng tăng'
body(sạch)  : 'Nội dung quan trọng & chi tiết.'
published_at: 2026-07-07T09:00:00+07:00 (naive -> gán UTC+7)
author/cat  : An / Kinh tế
tokenizer — tiếng Việt: normalize / fold dấu / tokenize
gốc          : Lạm phát ở Đồng Tháp TĂNG mạnh!
normalize    : lạm phát ở đồng tháp tăng mạnh!
fold_diacrit : Lam phat o Dong Thap TANG manh!   (Đồng -> Dong)
tokenize     : ['lạm', 'phát', 'ở', 'đồng', 'tháp', 'tăng', 'mạnh']
fold_tokens  : ['lam', 'phat', 'o', 'dong', 'thap', 'tang', 'manh']   (dùng cho match không dấu)


## 2. F-04 / F-09 — BM25 lexical: điểm số, match không dấu, trọng số tiêu đề

In [3]:
from news_search.index.lexical import LexicalIndex

def mk(i, t, b):
    return Article(i, t, b, f"/{i}", NOW)

lx = LexicalIndex()
lx.add(mk("d1", "Tin kinh tế", "lạm phát tăng khiến ngân hàng điều chỉnh lãi suất"))
lx.add(mk("d2", "Bản tin buổi chiều", "lạm phát tăng nhẹ trong quý một"))
lx.add(mk("d3", "Thể thao", "bóng đá trong nước có nhiều trận hay"))

h("search 'lạm phát ngân hàng' — d1 khớp cả 2 term -> điểm cao nhất")
for aid, sc in lx.search("lạm phát ngân hàng"):
    print(f"  {sc:6.3f}  {aid}")

h("search KHÔNG DẤU 'lam phat' -> vẫn ra bài có dấu (index folded)")
for aid, sc in lx.search("lam phat"):
    print(f"  {sc:6.3f}  {aid}")

h("Trọng số tiêu đề (title_weight=3): cùng term khớp ở tiêu đề > body")
lx2 = LexicalIndex()
lx2.add(mk("t1", "kinh tế việt nam", "nội dung nói về tăng trưởng chung"))
lx2.add(mk("t2", "bản tin buổi sáng", "kinh tế được nhắc tới tại đây"))
for aid, sc in lx2.search("kinh tế"):
    print(f"  {sc:6.3f}  {aid}   ({'khớp TIÊU ĐỀ' if aid=='t1' else 'khớp body'})")

search 'lạm phát ngân hàng' — d1 khớp cả 2 term -> điểm cao nhất
   2.781  d1
   0.901  d2
search KHÔNG DẤU 'lam phat' -> vẫn ra bài có dấu (index folded)
   0.901  d1
   0.901  d2
Trọng số tiêu đề (title_weight=3): cùng term khớp ở tiêu đề > body
   0.608  t1   (khớp TIÊU ĐỀ)
   0.365  t2   (khớp body)


## 3. F-15 — Snippet: bôi đậm từ khớp (query không dấu, giữ chữ có dấu)

In [4]:
from news_search.search.snippet import make_snippet

body = "Chỉ số lạm phát tháng sáu tăng mạnh, gây áp lực lên lãi suất ngân hàng nhà nước hôm nay."
h("query không dấu ['lam','phat','ngan','hang'] -> bôi đậm chữ CÓ DẤU")
print(make_snippet(body, ["lam", "phat", "ngan", "hang"], max_len=90))

h("Không match -> lấy đầu body; body dài -> có '...' ")
print(make_snippet(body, ["blockchain"], max_len=40))

query không dấu ['lam','phat','ngan','hang'] -> bôi đậm chữ CÓ DẤU
Chỉ số <b>lạm</b> <b>phát</b> tháng sáu tăng mạnh, gây áp lực lên lãi suất <b>ngân</b> <b>hàng</b> nhà nước hôm nay.
Không match -> lấy đầu body; body dài -> có '...' 
Chỉ số lạm phát tháng sáu tăng mạnh, gây...


## 4. F-05 / F-10 — Embedding + Vector index (cosine)

In [5]:
import numpy as np
from news_search.index.embeddings import HashingEmbedder
from news_search.index.vector import VectorIndex

emb = HashingEmbedder(dim=256)
V = emb.embed(["giá vàng tăng mạnh", "giá vàng đi lên", "đội tuyển bóng đá thắng lớn"])
h("HashingEmbedder — shape / dtype / L2-norm / cosine")
print("shape:", V.shape, "| dtype:", V.dtype)
print("norms (đã chuẩn hóa):", np.round(np.linalg.norm(V, axis=1), 4))
print("cosine(vàng1, vàng2)  =", round(float(V[0] @ V[1]), 3), " (gần nghĩa -> cao)")
print("cosine(vàng1, bóng đá) =", round(float(V[0] @ V[2]), 3), " (khác chủ đề -> thấp)")

h("VectorIndex — tìm láng giềng gần nhất")
vi = VectorIndex(256)
for i, t in enumerate(["giá vàng tăng", "bóng đá quốc gia", "lạm phát CPI tháng 6"]):
    vi.add(f"v{i}", emb.embed([t])[0])
q = emb.embed(["vàng tăng giá"])[0]
print("query 'vàng tăng giá' ->", [(a, round(s, 3)) for a, s in vi.search(q, top_k=3)])

HashingEmbedder — shape / dtype / L2-norm / cosine
shape: (3, 256) | dtype: float32
norms (đã chuẩn hóa): [1. 1. 1.]
cosine(vàng1, vàng2)  = 0.523  (gần nghĩa -> cao)
cosine(vàng1, bóng đá) = 0.367  (khác chủ đề -> thấp)
VectorIndex — tìm láng giềng gần nhất
query 'vàng tăng giá' -> [('v0', 0.784), ('v2', 0.322), ('v1', 0.315)]


## 5. F-06 — NER trích xuất thực thể + Knowledge Graph

In [6]:
from news_search.index.entities import extract_entities, KnowledgeGraph

sent = "Thủ tướng Phạm Minh Chính làm việc với UBND tỉnh Đồng Tháp và Tập đoàn Vingroup."
h("extract_entities")
print("Câu:", sent)
for e in extract_entities(sent):
    print(f"  {e.type:5} | {e.name}")

h("KnowledgeGraph — tra cứu theo tên KHÔNG DẤU")
kg = KnowledgeGraph()
kg.add_article("a1", extract_entities(sent))
kg.add_article("a2", [Entity("Phạm Minh Chính", "PER"), Entity("Hà Nội", "LOC")])
print("articles_for_entity('pham minh chinh'):", kg.articles_for_entity("pham minh chinh"))
print("articles_for_entity('HÀ NỘI')          :", kg.articles_for_entity("HÀ NỘI"))
print("top_entities:", [(e.name, c) for e, c in kg.top_entities(3)])

extract_entities
Câu: Thủ tướng Phạm Minh Chính làm việc với UBND tỉnh Đồng Tháp và Tập đoàn Vingroup.
  PER   | Phạm Minh Chính
  ORG   | UBND tỉnh Đồng Tháp
  ORG   | Tập đoàn Vingroup
KnowledgeGraph — tra cứu theo tên KHÔNG DẤU
articles_for_entity('pham minh chinh'): {'a2', 'a1'}
articles_for_entity('HÀ NỘI')          : {'a2'}
top_entities: [('Phạm Minh Chính', 2), ('Hà Nội', 1), ('Tập đoàn Vingroup', 1)]


## 6. F-07 — Phát hiện trùng/gần trùng (MinHash-LSH)

In [7]:
from news_search.index.dedup import MinHashDeduper

A = ("ủy ban bầu cử quốc gia công bố kết quả chính thức cuộc bầu cử tổng thống "
     "vào sáng nay tại thủ đô sau nhiều giờ kiểm phiếu căng thẳng")
B = A.replace("sáng", "chiều")          # gần trùng (khác 1 từ)
C = ("đội tuyển bóng đá quốc gia giành chiến thắng trong trận chung kết "
     "giải vô địch khu vực trước sự cổ vũ của khán giả")   # khác hẳn

d = MinHashDeduper(num_perm=128, threshold=0.6, shingle_size=3)
h("Gom cụm 3 văn bản (a,b gần trùng; c khác hẳn)")
for i, t in zip("abc", [A, B, C]):
    print(f"  {i} -> cluster {d.add(i, t)!r}")
print("sim(a,b) =", round(d.similarity("a", "b"), 3), " (gần trùng -> cao)")
print("sim(a,c) =", round(d.similarity("a", "c"), 3), " (khác hẳn -> thấp)")

Gom cụm 3 văn bản (a,b gần trùng; c khác hẳn)
  a -> cluster 'a'
  b -> cluster 'a'
  c -> cluster 'c'
sim(a,b) = 0.781  (gần trùng -> cao)
sim(a,c) = 0.0  (khác hẳn -> thấp)


## 7. Query Understanding — parse_query & nhận diện tin nóng (QDF)

In [8]:
from news_search.search.query import parse_query

h("parse_query — normalized / folded / tokens / fresh_intent")
for q in ["lạm phát tháng 6", "động đất hôm nay", "tin mới nhất về giá vàng",
          "ô nhiễm môi trường"]:
    p = parse_query(q)
    print(f"  {q!r:28} fresh_intent={str(p.fresh_intent):5} tokens={p.tokens}")
print("\nLưu ý: 'môi trường' (folded 'moi truong') KHÔNG bị nhận nhầm là 'mới'.")

parse_query — normalized / folded / tokens / fresh_intent
  'lạm phát tháng 6'           fresh_intent=False tokens=['lạm', 'phát', 'tháng', '6']
  'động đất hôm nay'           fresh_intent=True  tokens=['động', 'đất', 'hôm', 'nay']
  'tin mới nhất về giá vàng'   fresh_intent=True  tokens=['tin', 'mới', 'nhất', 'về', 'giá', 'vàng']
  'ô nhiễm môi trường'         fresh_intent=False tokens=['ô', 'nhiễm', 'môi', 'trường']

Lưu ý: 'môi trường' (folded 'moi truong') KHÔNG bị nhận nhầm là 'mới'.


## 8. F-11 — Reciprocal Rank Fusion (hybrid)

In [9]:
from news_search.search.fusion import rrf

lex = ["d1", "d2", "d3"]      # thứ hạng nhánh lexical
vec = ["d3", "d1", "d4"]      # thứ hạng nhánh vector
h("rrf([lexical, vector], k=60) — cộng dồn 1/(k+rank)")
for a, s in sorted(rrf([lex, vec], k=60).items(), key=lambda kv: -kv[1]):
    print(f"  {s:.5f}  {a}")
print("d1, d3 xuất hiện ở CẢ hai nhánh -> điểm cao nhất.")

rrf([lexical, vector], k=60) — cộng dồn 1/(k+rank)
  0.03252  d1
  0.03227  d3
  0.01613  d2
  0.01587  d4
d1, d3 xuất hiện ở CẢ hai nhánh -> điểm cao nhất.


## 9. F-13 — Time-decay / QDF: truy vấn thường (nhẹ) vs tin nóng (gắt)

In [10]:
from news_search.search.ranking import time_decay_multiplier

now = NOW
h("Hệ số suy giảm theo tuổi bài")
print(f"{'tuổi(ngày)':>11}{'thường (hl=30, floor=.5)':>26}{'tin nóng (hl=2, floor=.5)':>27}")
for age in [0, 0.25, 1, 2, 5, 10, 20, 30]:
    pub = now - timedelta(days=age)
    normal = time_decay_multiplier(pub, now, 30.0, 0.5)
    fresh = time_decay_multiplier(pub, now, 2.0, 0.5)
    print(f"{age:>11}{normal:>26.3f}{fresh:>27.3f}")
print("\nTruy vấn thường: bài cũ giảm nhẹ (không bị vùi). Tin nóng: recency áp đảo.")

Hệ số suy giảm theo tuổi bài
 tuổi(ngày)  thường (hl=30, floor=.5)  tin nóng (hl=2, floor=.5)
          0                     1.000                      1.000
       0.25                     0.997                      0.959
          1                     0.989                      0.854
          2                     0.977                      0.750
          5                     0.945                      0.588
         10                     0.897                      0.516
         20                     0.815                      0.500
         30                     0.750                      0.500

Truy vấn thường: bài cũ giảm nhẹ (không bị vùi). Tin nóng: recency áp đảo.


## 10. F-14 — Đa dạng hóa: gom cụm + MMR

In [11]:
import numpy as np
from news_search.search.diversify import collapse_clusters, mmr

h("collapse_clusters — mỗi cụm giữ tối đa 1 bài")
ranked = ["a", "b", "c", "d", "e"]
cluster = {"a": "C1", "b": "C1", "c": "C2", "d": None, "e": "C2"}
print("ranked   :", ranked)
print("cluster  :", cluster)
print("collapsed:", collapse_clusters(ranked, lambda i: cluster.get(i), 1),
      " (b trùng C1, e trùng C2 bị loại)")

h("MMR — chọn bài thứ 2 KHÁC HƯỚNG thay vì bản gần trùng")
vecs = {"i0": np.array([1., 0, 0]), "i1": np.array([0, 1., 0]), "i2": np.array([1., 0, 0])}
order = mmr([("i0", 1.0), ("i1", 0.9), ("i2", 0.8)], lambda i: vecs.get(i), 0.7, 3)
print("thứ tự chọn:", order, " (i1 khác hướng được ưu tiên trên i2 trùng hướng i0)")

collapse_clusters — mỗi cụm giữ tối đa 1 bài
ranked   : ['a', 'b', 'c', 'd', 'e']
cluster  : {'a': 'C1', 'b': 'C1', 'c': 'C2', 'd': None, 'e': 'C2'}
collapsed: ['a', 'c', 'd']  (b trùng C1, e trùng C2 bị loại)
MMR — chọn bài thứ 2 KHÁC HƯỚNG thay vì bản gần trùng
thứ tự chọn: ['i0', 'i1', 'i2']  (i1 khác hướng được ưu tiên trên i2 trùng hướng i0)


## 11. Pipeline end-to-end — 5 truy vấn nghiệm thu (BaoCao §3)

Mỗi kết quả in `[ngày] article_id  tiêu đề`. Quan sát: freshness, dedup, lọc metadata.

In [12]:
def show(label, q):
    h(label)
    res = pipeline.search(q)
    if not res:
        print("  (không có kết quả)")
    for r in res:
        print(f"  [{r.published_at[:10]}] {r.article_id:16} {r.title}")

show('Từ khóa: "lạm phát tháng 6"',
     SearchQuery("lạm phát tháng 6", top_k=5, now=NOW))
show('Câu hỏi tự nhiên: "vì sao giá vàng tăng mạnh tuần này"',
     SearchQuery("vì sao giá vàng tăng mạnh tuần này", top_k=5, now=NOW))
show('Tin nóng: "động đất" (bài 07/07 trước bài nền 26/06)',
     SearchQuery("động đất", top_k=5, now=NOW))
show('Lọc chuyên mục Kinh tế: "giá xăng"',
     SearchQuery("giá xăng", top_k=5, now=NOW, filters=SearchFilters(category="Kinh tế")))
show('Trùng lặp: "kết quả bầu cử" (3 bản tin gần trùng gom 1)',
     SearchQuery("kết quả bầu cử", top_k=8, now=NOW))

Từ khóa: "lạm phát tháng 6"
  [2026-07-06] eco-cpi-01       CPI tháng 6 tăng 3,2%, lạm phát trong tầm kiểm soát
  [2026-07-05] eco-cpi-02       Chuyên gia: áp lực lạm phát nửa cuối năm đến từ giá năng lượng
  [2026-07-05] eco-bank-01      Ngân hàng Nhà nước điều chỉnh lãi suất điều hành
  [2026-07-03] world-02         Căng thẳng thương mại leo thang giữa các nền kinh tế lớn
  [2026-07-03] sport-01         Đội tuyển bóng đá quốc gia thắng trận giao hữu
Câu hỏi tự nhiên: "vì sao giá vàng tăng mạnh tuần này"
  [2026-07-07] eco-gold-01      Giá vàng lập đỉnh: ba nguyên nhân chính đẩy giá đi lên
  [2026-07-06] eco-gold-02      Giá vàng trong nước sáng nay tăng theo giá thế giới
  [2026-07-07] news-quake-new   Động đất 5,1 độ tại Kon Tum, người dân cảm nhận rung lắc
  [2026-07-06] eco-cpi-01       CPI tháng 6 tăng 3,2%, lạm phát trong tầm kiểm soát
  [2026-07-04] soc-gas-02       Người dân xoay xở ra sao khi giá xăng biến động liên tục
Tin nóng: "động đất" (bài 07/07 trước bài nền 26/06)
  [

Xem chi tiết một kết quả (đủ 5 trường + snippet bôi đậm):

In [13]:
import json as _json
r = pipeline.search(SearchQuery("lạm phát tháng 6", top_k=1, now=NOW))[0]
print(_json.dumps(r.to_dict(), ensure_ascii=False, indent=2))
print("\nSố trường:", len(r.to_dict()), "->", sorted(r.to_dict().keys()))

{
  "article_id": "eco-cpi-01",
  "title": "CPI tháng 6 tăng 3,2%, lạm phát trong tầm kiểm soát",
  "url": "/kinh-te/cpi-thang-6-lam-phat-8821.html",
  "published_at": "2026-07-06T08:15:00+07:00",
  "snippet": "...iêu dùng (CPI) <b>tháng</b> <b>6</b> tăng 3,2% so với cùng kỳ năm trước. <b>Lạm</b> <b>phát</b> cơ bản được giữ ổn định nhờ nguồn cung hàng hóa dồi dào và giá lương thực hạ nhiệt. Giới chuyên gia đánh giá <b>lạm</b> <b>phát</b> vẫn nằm trong ..."
}

Số trường: 5 -> ['article_id', 'published_at', 'snippet', 'title', 'url']


## 12. F-08 — Đồng bộ gỡ bài (unpublish biến mất khỏi MỌI chỉ mục)

In [14]:
h("Trước khi gỡ")
print([r.article_id for r in pipeline.search(SearchQuery("giá vàng", top_k=10, now=NOW))])

manager.index_article({
    "article_id": "eco-gold-01", "title": "x", "body": "x", "url": "/x",
    "published_at": "2026-07-07T09:05:00+07:00", "status": "unpublished",
})
h("Sau khi unpublish 'eco-gold-01'")
print([r.article_id for r in pipeline.search(SearchQuery("giá vàng", top_k=10, now=NOW))])
print("Còn trong store? ", manager.store.get("eco-gold-01") is not None)
print("Còn trong lexical?", "eco-gold-01" in manager.lexical)
print("Còn vector?      ", manager.vector.get("eco-gold-01") is not None)
print("Còn cluster?     ", manager.deduper.cluster_of("eco-gold-01") is not None)

Trước khi gỡ
['eco-gold-01', 'eco-gold-02', 'elect-dup-1', 'soc-gas-02', 'eco-cpi-02', 'health-01', 'sport-01', 'eco-gas-01', 'cul-02', 'eco-cpi-01']
Sau khi unpublish 'eco-gold-01'
['eco-gold-02', 'elect-dup-1', 'soc-gas-02', 'eco-cpi-02', 'health-01', 'sport-01', 'cul-02', 'eco-gas-01', 'eco-cpi-01', 'tech-02']
Còn trong store?  False
Còn trong lexical? False
Còn vector?       False
Còn cluster?      False


## 13. Tầng API (FastAPI TestClient) — JSON 5 trường, mã lỗi, CRUD

In [15]:
from fastapi.testclient import TestClient
from news_search.api.app import create_app

app = create_app(Settings(embedder="hash", vector_backend="local"))
c = TestClient(app)
raws = json.load(open("data/sample_articles.json", encoding="utf-8"))
print("POST /articles/bulk ->", c.post("/articles/bulk", json=raws).json())
print("GET  /healthz       ->", c.get("/healthz").json())

h("GET /search?q=lam phat (KHÔNG DẤU) — mỗi phần tử đúng 5 trường")
for it in c.get("/search", params={"q": "lam phat", "top_k": 3}).json():
    print("  keys =", sorted(it.keys()), "->", it["article_id"])

h("Mã lỗi & CRUD")
print("q rỗng         ->", c.get("/search", params={"q": ""}).status_code, "(mong đợi 400)")
print("thiếu 'body'   ->", c.post("/articles", json={
    "article_id": "x", "title": "t", "url": "/x",
    "published_at": "2026-07-07T00:00:00+07:00"}).status_code, "(mong đợi 422)")
new = {"article_id": "live-1", "title": "Vệ tinh viễn thám mới",
       "body": "Vệ tinh viễn thám thế hệ mới được phóng lên quỹ đạo.",
       "url": "/kh/live-1", "published_at": "2026-07-07T10:00:00+07:00"}
print("POST bài mới   ->", c.post("/articles", json=new).json())
print("search thấy?   ->", any(d["article_id"] == "live-1"
      for d in c.get("/search", params={"q": "vệ tinh viễn thám"}).json()))
print("DELETE         ->", c.delete("/articles/live-1").json())
print("search sau xóa ->", any(d["article_id"] == "live-1"
      for d in c.get("/search", params={"q": "vệ tinh viễn thám"}).json()))
print("entities(elect-analysis):", c.get("/articles/elect-analysis/entities").json())

POST /articles/bulk -> {'indexed': 29}
GET  /healthz       -> {'status': 'ok', 'articles': 29, 'lexical': 29, 'vectors': 29, 'embedder': 'hash', 'vector_backend': 'local'}
GET /search?q=lam phat (KHÔNG DẤU) — mỗi phần tử đúng 5 trường
  keys = ['article_id', 'published_at', 'snippet', 'title', 'url'] -> eco-cpi-01
  keys = ['article_id', 'published_at', 'snippet', 'title', 'url'] -> eco-cpi-02
  keys = ['article_id', 'published_at', 'snippet', 'title', 'url'] -> eco-bank-01
Mã lỗi & CRUD
q rỗng         -> 400 (mong đợi 400)
thiếu 'body'   -> 422 (mong đợi 422)
POST bài mới   -> {'indexed': True, 'article_id': 'live-1'}
search thấy?   -> True
DELETE         -> {'removed': True}
search sau xóa -> False
entities(elect-analysis): []


D:\MEDIASOFT\news-search-engine\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 14. Benchmark — lexical vs dense vs hybrid (hiệu quả + tốc độ)

In [16]:
import time
from scripts import benchmark as bm

mgr = IndexManager(Settings(embedder="hash", vector_backend="local"))
mgr.bulk_index(json.load(open("data/sample_articles.json", encoding="utf-8")))
pl = SearchPipeline(mgr)
pmode = {"lexical": "lexical", "dense": "semantic", "hybrid": "hybrid"}

h(f"Trên {len(bm.QRELS)} truy vấn có nhãn (K=10)")
print(f"{'mode':8}{'Recall@10':>11}{'MRR':>8}{'nDCG@10':>9}{'p50(ms)':>10}")
for mode in ["lexical", "dense", "hybrid"]:
    rs = ms = ns = 0.0
    for q, rel in bm.QRELS.items():
        r = bm.retrieve(mgr, q, mode, 10)
        rs += bm.recall_at_k(r, rel, 10); ms += bm.mrr(r, rel); ns += bm.ndcg_at_k(r, rel, 10)
    N = len(bm.QRELS)
    lat = []
    for _ in range(5):
        for q in bm.QRELS:
            t = time.perf_counter()
            pl.search(SearchQuery(q, mode=pmode[mode], top_k=10, now=NOW))
            lat.append((time.perf_counter() - t) * 1000)
    lat.sort()
    print(f"{mode:8}{rs/N:>11.3f}{ms/N:>8.3f}{ns/N:>9.3f}{lat[len(lat)//2]:>10.2f}")
print("\n(Embedder hash: lexical mạnh trên corpus nhỏ; BGE-m3 thật -> dense cải thiện rõ.)")

Trên 10 truy vấn có nhãn (K=10)
mode      Recall@10     MRR  nDCG@10   p50(ms)


lexical       1.000   1.000    0.985      3.87


dense         0.950   1.000    0.953      7.18


hybrid        1.000   1.000    0.979      6.73

(Embedder hash: lexical mạnh trên corpus nhỏ; BGE-m3 thật -> dense cải thiện rõ.)
